In [1]:
import json 
with open("/home/turning/Jainit/TANQ/eval_dataset/tables.json") as f:
    tables = json.load(f)

In [2]:
import nltk
from nltk.translate.meteor_score import meteor_score
nltk.download('wordnet')

def linearize_table(table):
    """
    Convert a table (list of dictionaries) into a linearized string.
    Example: {"Name": "John", "Age": 30} → "Name: John; Age: 30"
    """
    linearized = []
    for row in table:
        row_str = "; ".join([f"{k}: {v}" for k, v in row.items()])
        linearized.append(row_str)
    return " ".join(linearized)

def compute_meteor_for_tables(reference_table, generated_table):
    """
    Compute METEOR score between a reference table and a generated table.
    """
    # Linearize tables into strings
    ref_str = linearize_table(reference_table)
    gen_str = linearize_table(generated_table)
    
    # Tokenize (split into words)
    reference = ref_str.split()
    hypothesis = gen_str.split()
    
    # Compute METEOR (alpha=0.9 weights recall more than precision)
    # print(reference)
    # print(hypothesis)
    return meteor_score([reference], hypothesis)


[nltk_data] Downloading package wordnet to /home/turning/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


In [3]:

def table_to_dict_list_comparison(table_string, suffix=""):
    table_string = table_string.replace("markdown", "")
    # Split the table into lines

    lines = table_string.strip().split('\n')

    # Find the first line that contains the table header (i.e., a line with '|')
    try:
        table_start_idx = next((i for i, line in enumerate(
            lines) if '|' in line), None)
    except:
        print("Error in table")
        print(table_string)

    # If no table is found, return an empty list
    if table_start_idx is None:
        return []

    # Process the header from the detected table start line
    header = lines[table_start_idx].strip().split('|')
    header = [col.strip() for col in header if col.strip()]

    # Prepare the list to hold dictionaries
    table_as_dicts = []

    # Loop through each data row, skipping any separator rows and stopping at ``` or blank lines
    for line in lines[table_start_idx + 1:]:
        # Stop processing if the table ends
        if '```' in line or not line.strip():
            break

        # Skip lines that contain only '---'
        if '---' in line:
            continue

        row_values = line.strip().split('|')
        row_values = [val.strip() for val in row_values if val.strip()]

        # Create a dictionary for the current row, ensuring to match header order with values
        row_dict = {}
        for i in range(len(header)):
            if i < len(row_values):
                row_dict[header[i]] = row_values[i]
            else:
                row_dict[header[i]] = None  # Fill with None if data is missing

        table_as_dicts.append(row_dict)

    return table_as_dicts


In [4]:
score_results = []
for table in tables: 
    scores = {}
    original_table = table["original"]
    for i in range(5):
        per_table = table[f'pertubation{i}']
        score = compute_meteor_for_tables(table_to_dict_list_comparison(original_table), table_to_dict_list_comparison(per_table))
        scores[f'result{i}'] = score
    score_results.append({**table, **scores})
    
    

In [10]:
with open("/home/turning/Jainit/TANQ/eval_dataset/old_metrics/results/meteor_scores.json", "w") as f:
    json.dump(score_results, f)